In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

In [3]:
load_dotenv()

True

In [4]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [5]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [6]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [7]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

In [8]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [9]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Elon Musk', 'June 28, 1971', 'Pretoria, South Africa', 'American', 'Entrepreneur', 'Engineer', "World'S Wealthiest Person"]
Rels:  [('Elon Musk', 'BORN_ON', 'June 28, 1971'), ('Elon Musk', 'BORN_IN', 'Pretoria, South Africa'), ('Elon Musk', 'HAS_NATIONALITY', 'American'), ('Elon Musk', 'HAS_OCCUPATION', 'Entrepreneur'), ('Elon Musk', 'HAS_OCCUPATION', 'Engineer'), ('Elon Musk', 'RECOGNISED_AS', "World'S Wealthiest Person")]


In [13]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

Graph stored in Neo4J


In [14]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [15]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 24}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Location'], 'count': 12}
  {'label': ['__Entity__', 'Organization'], 'count': 11}
  {'label': ['__Entity__', 'Product'], 'count': 10}
  {'label': ['__Entity__', 'Occupation'], 'count': 5}
  {'label': ['__Entity__', 'Date'], 'count': 4}
  {'label': ['__Entity__', 'Nationality'], 'count': 3}
  {'label': ['__Entity__', 'Year'], 'count': 3}
  {'label': ['__Entity__', 'Concept'], 'count': 3}
  {'label': ['__Entity__', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Service'], 'count': 2}
  {'label': ['__Entity__', 'Recognition'], 'count': 1}
  {'label': ['__Entity__', 'Software'], 'count': 1}
  {'label': ['__Entity__', 'Amount', 'Monetaryamount'], 'count': 1}
  {'label': ['__Entity__', 'Goal'], 'count': 1}
  {'label': ['__Entity__', 'Research'], 'count': 1}
  {'label': ['__Entity__', 'Thing'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 115}
  {'typ